In [1]:
import json

with open("../outputs/eda_summary.json", "r") as f:
    eda_summary = json.load(f)

eda_summary.keys()


dict_keys(['dataset_shape', 'target_column', 'task_type', 'numerical_columns', 'categorical_columns', 'missing_percentage', 'high_missing_columns', 'moderate_missing_columns', 'categorical_cardinality', 'high_cardinality_columns', 'skewness', 'high_skew_columns', 'correlated_pairs', 'target_distribution'])

In [2]:
preprocessing_plan = {
    "imputation": {},
    "encoding": {},
    "transformation": [],
    "scaling": [],
    "drop_columns": [],
}


| Missing % | Decision                 |
| --------- | ------------------------ |
| > 30%     | Drop or special handling |
| 5–30%     | Impute                   |
| < 5%      | Simple imputation        |


In [3]:
for col, pct in eda_summary["missing_percentage"].items():
    if pct > 30:
        preprocessing_plan["drop_columns"].append(col)

    elif pct > 5:
        preprocessing_plan["imputation"][col] = "median" \
            if col in eda_summary["numerical_columns"] else "most_frequent"

    elif pct > 0:
        preprocessing_plan["imputation"][col] = "mean" \
            if col in eda_summary["numerical_columns"] else "most_frequent"


| Cardinality | Encoding         |
| ----------- | ---------------- |
| ≤ 10        | OneHot           |
| > 10        | Target / Ordinal |


In [4]:
for col, card in eda_summary["categorical_cardinality"].items():
    if card <= 10:
        preprocessing_plan["encoding"][col] = "onehot"
    else:
        preprocessing_plan["encoding"][col] = "target"


|skew| > 1 → Power / log transform candidate

In [6]:
for col in eda_summary["high_skew_columns"]:
    preprocessing_plan["transformation"].append({
        "column": col,
        "method": "power"
    })


In [7]:
preprocessing_plan["scaling"] = [
    "standard",
    "minmax",
    "robust"
]


In [9]:
import json

with open("../outputs/preprocessing_plan.json", "w") as f:
    json.dump(preprocessing_plan, f, indent=4)

In [8]:
preprocessing_plan


{'imputation': {'Age': 'median', 'Embarked': 'most_frequent'},
 'encoding': {'Sex': 'onehot', 'Cabin': 'target', 'Embarked': 'onehot'},
 'transformation': [{'column': 'SibSp', 'method': 'power'},
  {'column': 'Parch', 'method': 'power'},
  {'column': 'Fare', 'method': 'power'},
  {'column': 'SibSp', 'method': 'power'},
  {'column': 'Parch', 'method': 'power'},
  {'column': 'Fare', 'method': 'power'}],
 'scaling': ['standard', 'minmax', 'robust'],
 'drop_columns': ['Cabin']}

Tadaa separates decision logic from execution, making preprocessing explainable, auditable, and reproducible.